# BioMedCLIP on BloodMNIST

A reproducible workflow for zero-shot classification, a frozen linear probe, and partial fine-tuning of BioMedCLIP on BloodMNIST.

The notebook keeps exactly one dataset object per split and uses `num_workers=0` for image loaders to control unified-memory use on Apple silicon. Expensive stages are controlled from the configuration section. Existing checkpoints and embedding caches are loaded by default.

# 1. Imports and reproducibility

Install dependencies in the active notebook environment only if they are missing:

```python
%pip install medmnist open_clip_torch transformers matplotlib seaborn scikit-learn pandas tqdm
```

Restart the kernel after installing packages, then begin here.

In [ ]:
import copy
import json
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import medmnist
import numpy as np
import open_clip
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from IPython.display import display
from PIL import Image
from medmnist import INFO
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

In [ ]:
def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for repeatable runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


seed_everything(42)

# 2. Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    seed: int = 42
    model_name: str = (
        "hf-hub:microsoft/"
        "BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
    image_size: int = 224
    embedding_size: int = 512
    image_batch_size: int = 4
    embedding_batch_size: int = 64
    num_workers: int = 0
    linear_epochs: int = 20
    partial_epochs: int = 5
    linear_lr: float = 1e-3
    visual_lr: float = 1e-5
    classifier_lr: float = 1e-4
    weight_decay: float = 1e-4

    # Safe defaults: reuse completed work instead of retraining.
    run_zero_shot: bool = False
    linear_probe_mode: str = "load"       # "load" or "train"
    partial_finetune_mode: str = "load"   # "load" or "train"
    evaluate_models: bool = False
    show_plots: bool = True
    extract_embeddings_if_missing: bool = False


CONFIG = Config()
seed_everything(CONFIG.seed)

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name != "FineTuning" and (PROJECT_DIR / "FineTuning").is_dir():
    PROJECT_DIR = PROJECT_DIR / "FineTuning"

PATHS = {
    "linear_checkpoint": PROJECT_DIR / "bloodmnist_linear_probe.pt",
    "embedding_cache": PROJECT_DIR / "bloodmnist_biomedclip_embeddings.pt",
    "test_embedding_cache": PROJECT_DIR / "bloodmnist_biomedclip_test_embeddings.pt",
    "partial_best": PROJECT_DIR / "bloodmnist_partial_finetune_best.pt",
    "partial_final": PROJECT_DIR / "bloodmnist_biomedclip_final.pt",
    "results_csv": PROJECT_DIR / "bloodmnist_results.csv",
    "run_manifest": PROJECT_DIR / "bloodmnist_run_manifest.json",
}

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Project directory:", PROJECT_DIR)
print("Device:", device)
print("Training modes:", CONFIG.linear_probe_mode, CONFIG.partial_finetune_mode)

# 3. Dataset creation

Each BloodMNIST split is instantiated exactly once. The datasets initially have no transform so the raw training images can be visualized. Section 5 assigns BioMedCLIP transforms and creates the image loaders.

In [ ]:
info = INFO["bloodmnist"]
DataClass = medmnist.BloodMNIST
class_names = [info["label"][str(i)] for i in range(len(info["label"]))]
number_of_classes = len(class_names)


def create_datasets(config: Config):
    common = {
        "download": True,
        "size": config.image_size,
        "as_rgb": True,
    }
    return {
        split: DataClass(split=split, transform=None, **common)
        for split in ("train", "val", "test")
    }


datasets = create_datasets(CONFIG)
train_dataset = datasets["train"]
val_dataset = datasets["val"]
test_dataset = datasets["test"]

for split, dataset in datasets.items():
    print(f"{split:>5}: {len(dataset):,} images")

In [ ]:
def create_image_loaders(datasets, config: Config):
    """Create loaders that all reference the three shared dataset objects."""
    return {
        "train": DataLoader(
            datasets["train"],
            batch_size=config.image_batch_size,
            shuffle=True,
            num_workers=config.num_workers,
        ),
        "val": DataLoader(
            datasets["val"],
            batch_size=config.image_batch_size,
            shuffle=False,
            num_workers=config.num_workers,
        ),
        "test": DataLoader(
            datasets["test"],
            batch_size=config.image_batch_size,
            shuffle=False,
            num_workers=config.num_workers,
        ),
    }

# 4. Data visualization

In [ ]:
def raw_rgb_image(dataset, index: int) -> Image.Image:
    """Read a source image without depending on the dataset's active transform."""
    return Image.fromarray(dataset.imgs[index]).convert("RGB")


def plot_examples(dataset, names, rows=3, columns=4, seed=42):
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), rows * columns)
    fig, axes = plt.subplots(rows, columns, figsize=(12, 9))
    for index, ax in zip(indices, axes.flat):
        label_number = int(dataset.labels[index].item())
        ax.imshow(raw_rgb_image(dataset, index))
        ax.set_title(names[label_number])
        ax.axis("off")
    fig.tight_layout()
    return fig


def plot_class_distribution(dataset, names):
    labels = dataset.labels.reshape(-1)
    counts = np.bincount(labels, minlength=len(names))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(names, counts)
    ax.set(title="BloodMNIST training distribution", ylabel="Images")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    return fig


if CONFIG.show_plots:
    plot_examples(train_dataset, class_names, seed=CONFIG.seed)
    plt.show()
    plot_class_distribution(train_dataset, class_names)
    plt.show()

# 5. BioMedCLIP loading

In [ ]:
def load_biomedclip(model_name: str, target_device: torch.device):
    model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
        model_name
    )
    tokenizer = open_clip.get_tokenizer(model_name)
    model = model.to(target_device)
    model.eval()
    return model, tokenizer, preprocess_train, preprocess_val


model, tokenizer, preprocess_train, preprocess_val = load_biomedclip(
    CONFIG.model_name, device
)

# One dataset per split; transforms are properties of those shared objects.
train_dataset.transform = preprocess_train
val_dataset.transform = preprocess_val
test_dataset.transform = preprocess_val

image_loaders = create_image_loaders(datasets, CONFIG)
train_loader = image_loaders["train"]
val_loader = image_loaders["val"]
test_loader = image_loaders["test"]

print("Model device:", next(model.parameters()).device)
print("Image batch size:", CONFIG.image_batch_size)
print("Image-loader workers:", CONFIG.num_workers)

# 6. Zero-shot baseline

The second prompt set repeats the earlier experiment that shortened the long immature-granulocyte label. In the completed run, shortening it reduced validation accuracy from 27.63% to 20.27% and balanced accuracy from 19.96% to 14.31%.

In [ ]:
def make_prompts(names):
    return [f"a microscope image of a {name}" for name in names]


def encode_prompts(model, tokenizer, prompts, target_device):
    tokens = tokenizer(prompts, context_length=256).to(target_device)
    with torch.inference_mode():
        features = model.encode_text(tokens).float()
    return F.normalize(features, dim=1)


def collect_metrics(labels, predictions):
    labels = torch.as_tensor(labels).cpu().numpy()
    predictions = torch.as_tensor(predictions).cpu().numpy()
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "confusion_matrix": confusion_matrix(labels, predictions),
        "labels": labels,
        "predictions": predictions,
    }


def evaluate_zero_shot(model, loader, text_features, target_device):
    predictions, labels = [], []
    model.eval()
    with torch.inference_mode():
        for images, batch_labels in tqdm(loader, desc="Zero-shot"):
            images = images.to(target_device)
            image_features = F.normalize(
                model.encode_image(images).float(), dim=1
            )
            batch_predictions = (image_features @ text_features.T).argmax(dim=1)
            predictions.append(batch_predictions.cpu())
            labels.append(batch_labels.view(-1).long().cpu())
    return collect_metrics(torch.cat(labels), torch.cat(predictions))


runtime_results = []
standard_prompts = make_prompts(class_names)
short_class_names = class_names.copy()
short_class_names[3] = "immature granulocyte"
short_prompts = make_prompts(short_class_names)

if CONFIG.run_zero_shot:
    standard_text_features = encode_prompts(
        model, tokenizer, standard_prompts, device
    )
    short_text_features = encode_prompts(
        model, tokenizer, short_prompts, device
    )
    standard_zero_shot = evaluate_zero_shot(
        model, val_loader, standard_text_features, device
    )
    shortened_zero_shot = evaluate_zero_shot(
        model, val_loader, short_text_features, device
    )
    runtime_results.extend([
        {
            "method": "Zero-shot: original labels",
            "split": "validation",
            **{k: standard_zero_shot[k] for k in ("accuracy", "balanced_accuracy")},
        },
        {
            "method": "Zero-shot: shortened label",
            "split": "validation",
            **{k: shortened_zero_shot[k] for k in ("accuracy", "balanced_accuracy")},
        },
    ])
    print("Original-label accuracy:", standard_zero_shot["accuracy"])
    print("Shortened-label accuracy:", shortened_zero_shot["accuracy"])
else:
    print("Skipped zero-shot inference; set CONFIG.run_zero_shot=True to run it.")

# 7. Frozen linear probe

BioMedCLIP remains frozen while a `Linear(512, 8)` classifier learns from cached image embeddings. Existing embedding files and the verified best classifier checkpoint are loaded by default.

In [ ]:
def extract_embeddings(model, loader, target_device):
    embeddings, labels = [], []
    model.eval()
    with torch.inference_mode():
        for images, batch_labels in tqdm(loader, desc="Embeddings"):
            image_features = F.normalize(
                model.encode_image(images.to(target_device)).float(), dim=1
            )
            embeddings.append(image_features.cpu())
            labels.append(batch_labels.view(-1).long().cpu())
    return torch.cat(embeddings), torch.cat(labels)


def load_embedding_caches(paths):
    train_val = torch.load(
        paths["embedding_cache"], map_location="cpu", weights_only=True
    )
    test = torch.load(
        paths["test_embedding_cache"], map_location="cpu", weights_only=True
    )
    return {
        "train": (train_val["train_embeddings"], train_val["train_labels"]),
        "val": (train_val["val_embeddings"], train_val["val_labels"]),
        "test": (test["test_embeddings"], test["test_labels"]),
    }


def create_and_save_embedding_caches(model, image_loaders, datasets, paths, device):
    """Extract deterministic embeddings while retaining one dataset per split."""
    previous_train_transform = datasets["train"].transform
    datasets["train"].transform = preprocess_val
    try:
        train_embeddings, train_labels = extract_embeddings(
            model, image_loaders["train"], device
        )
        val_embeddings, val_labels = extract_embeddings(
            model, image_loaders["val"], device
        )
        test_embeddings, test_labels = extract_embeddings(
            model, image_loaders["test"], device
        )
    finally:
        datasets["train"].transform = previous_train_transform

    torch.save({
        "train_embeddings": train_embeddings,
        "train_labels": train_labels,
        "val_embeddings": val_embeddings,
        "val_labels": val_labels,
    }, paths["embedding_cache"])
    torch.save({
        "test_embeddings": test_embeddings,
        "test_labels": test_labels,
    }, paths["test_embedding_cache"])
    return {
        "train": (train_embeddings, train_labels),
        "val": (val_embeddings, val_labels),
        "test": (test_embeddings, test_labels),
    }


cache_files_exist = all(
    PATHS[key].is_file()
    for key in ("embedding_cache", "test_embedding_cache")
)
if cache_files_exist:
    embedding_data = load_embedding_caches(PATHS)
elif CONFIG.extract_embeddings_if_missing:
    embedding_data = create_and_save_embedding_caches(
        model, image_loaders, datasets, PATHS, device
    )
else:
    raise FileNotFoundError(
        "Embedding cache is missing. Set extract_embeddings_if_missing=True "
        "to create it."
    )

embedding_loaders = {
    split: DataLoader(
        TensorDataset(embeddings, labels),
        batch_size=CONFIG.embedding_batch_size,
        shuffle=(split == "train"),
    )
    for split, (embeddings, labels) in embedding_data.items()
}

for split, (embeddings, labels) in embedding_data.items():
    print(split, tuple(embeddings.shape), tuple(labels.shape))

In [ ]:
def save_linear_probe(classifier, path, class_names, best_accuracy):
    payload = {
        "classifier_state_dict": {
            name: tensor.detach().cpu().clone()
            for name, tensor in classifier.state_dict().items()
        },
        "embedding_size": classifier.in_features,
        "number_of_classes": classifier.out_features,
        "class_names": list(class_names),
        "best_val_accuracy": float(best_accuracy),
    }
    torch.save(payload, path)


def load_linear_probe(path, expected_names, target_device):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if list(checkpoint["class_names"]) != list(expected_names):
        raise ValueError("Checkpoint class order does not match this dataset")
    classifier = nn.Linear(
        checkpoint["embedding_size"], checkpoint["number_of_classes"]
    )
    classifier.load_state_dict(checkpoint["classifier_state_dict"], strict=True)
    return classifier.to(target_device), checkpoint


def train_linear_probe(train_loader, val_loader, config, class_names, path, device):
    classifier = nn.Linear(config.embedding_size, len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        classifier.parameters(), lr=config.linear_lr,
        weight_decay=config.weight_decay,
    )
    best_accuracy = -1.0
    best_state = None
    history = []

    for epoch in range(config.linear_epochs):
        epoch_record = {"epoch": epoch + 1}
        for split, loader in (("train", train_loader), ("val", val_loader)):
            classifier.train(split == "train")
            total_loss = correct = total = 0
            with torch.set_grad_enabled(split == "train"):
                for embeddings, labels in loader:
                    embeddings = embeddings.to(device)
                    labels = labels.to(device)
                    if split == "train":
                        optimizer.zero_grad(set_to_none=True)
                    outputs = classifier(embeddings)
                    loss = criterion(outputs, labels)
                    if split == "train":
                        loss.backward()
                        optimizer.step()
                    total_loss += loss.item() * labels.size(0)
                    correct += (outputs.argmax(1) == labels).sum().item()
                    total += labels.size(0)
            epoch_record[f"{split}_loss"] = total_loss / total
            epoch_record[f"{split}_accuracy"] = correct / total
        history.append(epoch_record)
        if epoch_record["val_accuracy"] > best_accuracy:
            best_accuracy = epoch_record["val_accuracy"]
            best_state = copy.deepcopy(classifier.state_dict())

    classifier.load_state_dict(best_state)
    save_linear_probe(classifier, path, class_names, best_accuracy)
    return classifier, pd.DataFrame(history), best_accuracy


if CONFIG.linear_probe_mode == "train":
    linear_classifier, linear_history, linear_best_accuracy = train_linear_probe(
        embedding_loaders["train"], embedding_loaders["val"],
        CONFIG, class_names, PATHS["linear_checkpoint"], device,
    )
elif CONFIG.linear_probe_mode == "load":
    linear_classifier, linear_checkpoint = load_linear_probe(
        PATHS["linear_checkpoint"], class_names, device
    )
    linear_history = None
    linear_best_accuracy = linear_checkpoint.get("best_val_accuracy")
else:
    raise ValueError("linear_probe_mode must be 'load' or 'train'")

print("Linear probe best validation accuracy:", linear_best_accuracy)
print("Linear probe device:", next(linear_classifier.parameters()).device)

# 8. Partial fine-tuning

The visual encoder is frozen except for transformer block 11, the final visual normalization, and the 768→512 projection. The trained linear probe initializes the classifier. The optimizer is constructed only after both model and classifier are on the selected device.

In [ ]:
def configure_partial_finetuning(model):
    for parameter in model.parameters():
        parameter.requires_grad = False
    for module in (
        model.visual.trunk.blocks[-1],
        model.visual.trunk.norm,
        model.visual.head,
    ):
        for parameter in module.parameters():
            parameter.requires_grad = True
    return [
        parameter for parameter in model.visual.parameters()
        if parameter.requires_grad
    ]


def partial_checkpoint_payload(model, classifier, class_names, config, **metrics):
    cpu_state = lambda module: {
        name: tensor.detach().cpu().clone()
        for name, tensor in module.state_dict().items()
    }
    return {
        "block_11_state_dict": cpu_state(model.visual.trunk.blocks[-1]),
        "norm_state_dict": cpu_state(model.visual.trunk.norm),
        "visual_head_state_dict": cpu_state(model.visual.head),
        "classifier_state_dict": cpu_state(classifier),
        "model_name": config.model_name,
        "class_names": list(class_names),
        "image_size": config.image_size,
        "unfrozen_blocks": [11],
        "visual_learning_rate": config.visual_lr,
        "classifier_learning_rate": config.classifier_lr,
        "batch_size": config.image_batch_size,
        **metrics,
    }


def load_partial_checkpoint(model, classifier, path, target_device):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model.visual.trunk.blocks[-1].load_state_dict(
        checkpoint["block_11_state_dict"], strict=True
    )
    model.visual.trunk.norm.load_state_dict(
        checkpoint["norm_state_dict"], strict=True
    )
    model.visual.head.load_state_dict(
        checkpoint["visual_head_state_dict"], strict=True
    )
    classifier.load_state_dict(checkpoint["classifier_state_dict"], strict=True)
    return model.to(target_device), classifier.to(target_device), checkpoint


model = model.to(device)
partial_classifier = nn.Linear(CONFIG.embedding_size, number_of_classes)
partial_classifier.load_state_dict(
    {name: tensor.detach().cpu() for name, tensor in linear_classifier.state_dict().items()}
)
partial_classifier = partial_classifier.to(device)
trainable_visual_parameters = configure_partial_finetuning(model)

print(
    "Trainable visual parameters:",
    f"{sum(p.numel() for p in trainable_visual_parameters):,}",
)
print(
    "Trainable classifier parameters:",
    f"{sum(p.numel() for p in partial_classifier.parameters()):,}",
)

In [ ]:
def set_partial_train_mode(model, classifier):
    model.eval()
    model.visual.trunk.blocks[-1].train()
    model.visual.trunk.norm.train()
    model.visual.head.train()
    classifier.train()


def evaluate_image_classifier(model, classifier, loader, device, criterion=None, desc="Evaluate"):
    model.eval()
    classifier.eval()
    predictions, labels = [], []
    total_loss = total = 0
    with torch.inference_mode():
        for images, batch_labels in tqdm(loader, desc=desc):
            images = images.to(device)
            batch_labels = batch_labels.view(-1).long().to(device)
            features = F.normalize(model.encode_image(images).float(), dim=1)
            outputs = classifier(features)
            if criterion is not None:
                total_loss += criterion(outputs, batch_labels).item() * batch_labels.size(0)
            total += batch_labels.size(0)
            predictions.append(outputs.argmax(1).cpu())
            labels.append(batch_labels.cpu())
    metrics = collect_metrics(torch.cat(labels), torch.cat(predictions))
    metrics["loss"] = total_loss / total if criterion is not None else None
    return metrics


def train_partial_finetune(model, classifier, loaders, config, path, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW([
        {"params": [p for p in model.visual.parameters() if p.requires_grad],
         "lr": config.visual_lr},
        {"params": classifier.parameters(), "lr": config.classifier_lr},
    ], weight_decay=config.weight_decay)
    best_accuracy = -1.0
    history = []

    for epoch in range(config.partial_epochs):
        set_partial_train_mode(model, classifier)
        train_loss = train_correct = train_total = 0
        for images, labels in tqdm(loaders["train"], desc=f"Epoch {epoch + 1} train"):
            images = images.to(device)
            labels = labels.view(-1).long().to(device)
            optimizer.zero_grad(set_to_none=True)
            features = F.normalize(model.encode_image(images).float(), dim=1)
            outputs = classifier(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * labels.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        val_metrics = evaluate_image_classifier(
            model, classifier, loaders["val"], device, criterion, "Validation"
        )
        record = {
            "epoch": epoch + 1,
            "train_loss": train_loss / train_total,
            "train_accuracy": train_correct / train_total,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        }
        history.append(record)
        print(record)

        if val_metrics["accuracy"] > best_accuracy:
            best_accuracy = val_metrics["accuracy"]
            torch.save(
                partial_checkpoint_payload(
                    model, classifier, class_names, config,
                    best_val_accuracy=best_accuracy,
                    best_val_balanced_accuracy=val_metrics["balanced_accuracy"],
                ),
                path,
            )

    model, classifier, checkpoint = load_partial_checkpoint(
        model, classifier, path, device
    )
    return model, classifier, pd.DataFrame(history), checkpoint


if CONFIG.partial_finetune_mode == "train":
    model, partial_classifier, partial_history, partial_checkpoint = train_partial_finetune(
        model, partial_classifier, image_loaders, CONFIG, PATHS["partial_best"], device
    )
elif CONFIG.partial_finetune_mode == "load":
    source = PATHS["partial_final"] if PATHS["partial_final"].is_file() else PATHS["partial_best"]
    model, partial_classifier, partial_checkpoint = load_partial_checkpoint(
        model, partial_classifier, source, device
    )
    partial_history = None
else:
    raise ValueError("partial_finetune_mode must be 'load' or 'train'")

print("Partial checkpoint validation accuracy:", partial_checkpoint["best_val_accuracy"])

# 9. Validation and test evaluation

In [ ]:
def evaluate_embedding_classifier(classifier, loader, target_device):
    classifier.eval()
    predictions, labels = [], []
    with torch.inference_mode():
        for embeddings, batch_labels in loader:
            outputs = classifier(embeddings.to(target_device))
            predictions.append(outputs.argmax(1).cpu())
            labels.append(batch_labels.cpu())
    return collect_metrics(torch.cat(labels), torch.cat(predictions))


def print_evaluation(name, metrics, names):
    print(f"{name} accuracy: {metrics['accuracy']:.4f}")
    print(f"{name} balanced accuracy: {metrics['balanced_accuracy']:.4f}")
    print(classification_report(
        metrics["labels"], metrics["predictions"],
        target_names=names, digits=4,
    ))


def plot_confusion(metrics, names, title):
    matrix = confusion_matrix(
        metrics["labels"], metrics["predictions"], normalize="true"
    )
    fig, ax = plt.subplots(figsize=(11, 8))
    sns.heatmap(
        matrix, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=names, yticklabels=names, ax=ax,
    )
    ax.set(xlabel="Predicted", ylabel="True", title=title)
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    return fig


if CONFIG.evaluate_models:
    for split in ("val", "test"):
        linear_metrics = evaluate_embedding_classifier(
            linear_classifier, embedding_loaders[split], device
        )
        partial_metrics = evaluate_image_classifier(
            model, partial_classifier, image_loaders[split], device,
            nn.CrossEntropyLoss(), f"Partial {split}",
        )
        runtime_results.extend([
            {
                "method": "Frozen linear probe", "split": split,
                **{k: linear_metrics[k] for k in ("accuracy", "balanced_accuracy")},
            },
            {
                "method": "Partial fine-tuning", "split": split,
                **{k: partial_metrics[k] for k in ("accuracy", "balanced_accuracy")},
            },
        ])
        print_evaluation(f"Linear probe {split}", linear_metrics, class_names)
        print_evaluation(f"Partial fine-tuning {split}", partial_metrics, class_names)
else:
    print("Skipped evaluation; set CONFIG.evaluate_models=True to recompute metrics.")

# 10. Results comparison

These are the metrics from the completed run preserved in `BioMedCLIPFineTuning_completed_run.ipynb`. Runtime evaluations replace the matching rows when enabled.

| Method | Split | Accuracy | Balanced accuracy |
|---|---:|---:|---:|
| Zero-shot, original labels | Validation | 0.2763 | 0.1996 |
| Zero-shot, shortened label | Validation | 0.2027 | 0.1431 |
| Frozen linear probe | Validation | 0.8914 | 0.8666 |
| Frozen linear probe | Test | 0.8889 | 0.8574 |
| Partial fine-tuning | Validation | 0.9702 | — |
| Partial fine-tuning | Test | 0.9699 | 0.9642 |

The shortened label did not help: validation accuracy fell by 7.36 percentage points and balanced accuracy fell by 5.65 points.

In [ ]:
completed_run_results = [
    {"method": "Zero-shot: original labels", "split": "validation",
     "accuracy": 0.2763, "balanced_accuracy": 0.1996},
    {"method": "Zero-shot: shortened label", "split": "validation",
     "accuracy": 0.2027, "balanced_accuracy": 0.1431},
    {"method": "Frozen linear probe", "split": "validation",
     "accuracy": 0.8913551402, "balanced_accuracy": 0.8666},
    {"method": "Frozen linear probe", "split": "test",
     "accuracy": 0.8889, "balanced_accuracy": 0.8574},
    {"method": "Partial fine-tuning", "split": "validation",
     "accuracy": 0.9702102804, "balanced_accuracy": np.nan},
    {"method": "Partial fine-tuning", "split": "test",
     "accuracy": 0.9698918445, "balanced_accuracy": 0.9641613390},
]

results_by_key = {
    (row["method"], row["split"]): row
    for row in completed_run_results
}
for row in runtime_results:
    results_by_key[(row["method"], row["split"])] = row

results_df = (
    pd.DataFrame(results_by_key.values())
    .sort_values(["split", "accuracy"], ascending=[True, False])
    .reset_index(drop=True)
)
display(results_df.style.format({
    "accuracy": "{:.4f}",
    "balanced_accuracy": "{:.4f}",
}, na_rep="—"))

# 11. Saving and restoring checkpoints

The workflow saves compact state dictionaries rather than duplicating the entire BioMedCLIP model. Restore the original BioMedCLIP architecture first, then apply the partial checkpoint.

In [ ]:
def verify_checkpoint_files(paths):
    rows = []
    for name, path in paths.items():
        if path.suffix in {".pt", ".csv", ".json"}:
            rows.append({
                "artifact": name,
                "exists": path.is_file(),
                "path": str(path),
                "size_mb": path.stat().st_size / 1024**2 if path.is_file() else np.nan,
            })
    return pd.DataFrame(rows)


def save_results_and_manifest(results, config, paths):
    results.to_csv(paths["results_csv"], index=False)
    manifest = {
        "config": asdict(config),
        "artifacts": {name: str(path) for name, path in paths.items()},
        "results": results.where(pd.notna(results), None).to_dict("records"),
    }
    paths["run_manifest"].write_text(json.dumps(manifest, indent=2))
    return paths["results_csv"], paths["run_manifest"]


results_path, manifest_path = save_results_and_manifest(
    results_df, CONFIG, PATHS
)
print("Saved results:", results_path)
print("Saved run manifest:", manifest_path)
display(verify_checkpoint_files(PATHS))

In [ ]:
# Fresh restore example: classifier checkpoint only (no training required).
restored_linear_classifier, restored_linear_metadata = load_linear_probe(
    PATHS["linear_checkpoint"], class_names, torch.device("cpu")
)
assert restored_linear_classifier.in_features == CONFIG.embedding_size
assert restored_linear_classifier.out_features == number_of_classes
print(
    "Verified linear checkpoint; best validation accuracy:",
    restored_linear_metadata.get("best_val_accuracy"),
)

# To restore partial fine-tuning after a restart, first run sections 1–5,
# construct a 512→8 classifier, configure the trainable modules, then run:
# model, partial_classifier, metadata = load_partial_checkpoint(
#     model, partial_classifier, PATHS["partial_final"], device
# )